In [2]:
# ==============================================================================
# CRIME & CYBERCRIME MULTI-CLASS NLP CLASSIFIER (MODEL v3)
# Architecture: Pretrained DistilBERT (distilbert-base-uncased)
# Categories: 11 Standardized Clean Classes (Cyber & Physical)
# Dataset Size: 13,200 Balanced Records (1,200 samples/class)
# Hardware: Google Colab T4 GPU (~4 to 5 minutes)
# ==============================================================================

# Step 1: Install Required Libraries
!pip install -q transformers datasets evaluate accelerate scikit-learn

import os
import re
import shutil
import time
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

# ------------------------------------------------------------------------------
# 1. Device Verification
# ------------------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using Device: {device}")
if torch.cuda.is_available():
    print(f"[INFO] GPU Model: {torch.cuda.get_device_name(0)}")
else:
    print("[WARNING] GPU not detected! Go to Runtime > Change runtime type > T4 GPU.")

# ------------------------------------------------------------------------------
# 2. Load Crime Dataset v3 (13,200 Records)
# ------------------------------------------------------------------------------
CSV_FILE = "crime_dataset_v3.csv"

# Agar file upload nahi hui to prompt aayega
if not os.path.exists(CSV_FILE):
    from google.colab import files
    print("Please upload 'crime_dataset_v3.csv' from your PC:")
    uploaded = files.upload()

df = pd.read_csv(CSV_FILE)
print(f"\n[INFO] Loaded Records: {len(df):,}")
print(f"[INFO] Total Classes: {df['category'].nunique()}")
print("\nClass Distribution:")
print(df['category'].value_counts())

# ------------------------------------------------------------------------------
# 3. Label Encoding & Stratified Split
# ------------------------------------------------------------------------------
unique_labels = sorted(df['category'].unique().tolist())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}
df['label'] = df['category'].map(label2id)

# 85% Train, 15% Validation (Stratified split for equal representation)
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df['label']
)

MODEL_NAME = "distilbert-base-uncased"
print(f"\n[INFO] Loading Pretrained Tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenize_batch, batched=True)
val_dataset = Dataset.from_pandas(val_df[['text', 'label']]).map(tokenize_batch, batched=True)

print(f"[INFO] Training Samples: {len(train_dataset):,} | Validation Samples: {len(val_dataset):,}")

# ------------------------------------------------------------------------------
# 4. Model Architecture & Training Setup
# ------------------------------------------------------------------------------
print(f"[INFO] Initializing DistilBERT Classifier for {len(unique_labels)} classes...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "macro_f1": macro_f1, "weighted_f1": weighted_f1}

training_args = TrainingArguments(
    output_dir="./crime_v3_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),  # GPU acceleration
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

# ------------------------------------------------------------------------------
# 5. Start Training
# ------------------------------------------------------------------------------
print("\n" + "=" * 65)
print("             STARTING DISTILBERT v3 FINE-TUNING")
print("=" * 65)
start_time = time.time()
trainer.train()
duration = (time.time() - start_time) / 60
print(f"\n[SUCCESS] Training finished in {duration:.2f} minutes!")

# ------------------------------------------------------------------------------
# 6. Detailed Evaluation Report
# ------------------------------------------------------------------------------
print("\n[INFO] Running Validation Evaluation...")
val_preds = trainer.predict(val_dataset)
preds = np.argmax(val_preds.predictions, axis=-1)
true_labels = val_preds.label_ids

acc = accuracy_score(true_labels, preds) * 100
macro_f1 = f1_score(true_labels, preds, average="macro") * 100
weighted_f1 = f1_score(true_labels, preds, average="weighted") * 100

print("\n" + "=" * 65)
print(f"  OVERALL ACCURACY : {acc:.2f}%")
print(f"  MACRO F1-SCORE   : {macro_f1:.2f}%")
print(f"  WEIGHTED F1-SCORE: {weighted_f1:.2f}%")
print("=" * 65)

print("\nClassification Report for all 11 Categories:")
print(classification_report(true_labels, preds, target_names=unique_labels, digits=4))

# ------------------------------------------------------------------------------
# 7. Live Test Cases (Physical & Cyber)
# ------------------------------------------------------------------------------
test_cases = [
    "Two armed men on a motorcycle intercepted me at gunpoint near the market and snatched my wallet and iPhone.",
    "Returned home to find front door broken and gold jewellery and cash missing from the bedroom safe.",
    "A mob attacked my brother with an iron pipe outside our house causing severe head fracture and injuries.",
    "An unknown man has been persistently following my sister every evening from the university metro station.",
    "Received a threatening letter wrapped with a bullet demanding 500,000 PKR bhatta protection money for our shop.",
    "Received a fake bank SMS with an OTP link, resulting in 65,000 RS stolen from my bank account.",
    "Someone created a fake Instagram profile with my photos and is sending defamatory abusive messages to my relatives.",
    "All our office server databases were encrypted with .locked extension and the hacker demands 2 Bitcoin ransom.",
    "Invested 5,000 USDT in a Telegram automated crypto trading scheme, now the group is deleted and funds are stolen.",
    "Unauthorized hacker breached our company cloud server and leaked customer records online."
]

print("\n" + "=" * 65)
print("           LIVE PREDICTION TESTS (MODEL INFERENCE)")
print("=" * 65)
model.eval()
for sample in test_cases:
    inputs = tokenizer(sample, return_tensors="pt", truncation=True, max_length=128).to(model.device)
    inputs.pop("token_type_ids", None)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1)[0]
        pred_idx = torch.argmax(probs).item()
        confidence = probs[pred_idx].item() * 100

    print(f"Input : \"{sample}\"")
    print(f"Output: [{id2label[pred_idx]}] (Confidence: {confidence:.1f}%)\n")

# ------------------------------------------------------------------------------
# 8. Export Model & 1-Click Download Zip
# ------------------------------------------------------------------------------
EXPORT_DIR = "./crime_classifier_v3"
os.makedirs(EXPORT_DIR, exist_ok=True)

model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

shutil.make_archive("crime_classifier_v3", "zip", EXPORT_DIR)

print("=" * 65)
print("[SUCCESS] Model files saved and zipped to: crime_classifier_v3.zip")
print("=" * 65)

try:
    from google.colab import files
    files.download("crime_classifier_v3.zip")
    print("[INFO] Automatic browser download triggered!")
except Exception:
    print("[INFO] You can download 'crime_classifier_v3.zip' from the left Colab Files panel.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
[INFO] Using Device: cuda
[INFO] GPU Model: Tesla T4
Please upload 'crime_dataset_v3.csv' from your PC:


Saving crime_dataset_v3.csv to crime_dataset_v3.csv

[INFO] Loaded Records: 13,200
[INFO] Total Classes: 11

Class Distribution:
category
Hacking & Unauthorized Access     1200
Assault & Physical Violence       1200
Robbery & Armed Snatching         1200
Physical Harassment & Stalking    1200
Physical Extortion & Blackmail    1200
Cryptocurrency Fraud              1200
Financial Fraud                   1200
Ransomware & Digital Extortion    1200
Identity Theft & Impersonation    1200
Burglary & Break-in               1200
Cyber Harassment & Stalking       1200
Name: count, dtype: int64

[INFO] Loading Pretrained Tokenizer: distilbert-base-uncased


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[SUCCESS] Model files saved and zipped to: crime_classifier_v3.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[INFO] Automatic browser download triggered!


In [5]:
from google.colab import files

files.download("crime_classifier_v3.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>